# Feature Preview Notebook

This notebook provides a quick, practical view of the engineered feature set produced by the feature engineering stage.

The goals are to:

- inspect the final feature table shape and column groups,
- review the main engineered features by category,
- check for obvious quality issues such as missing values or duplicates, and
- get an early look at how features relate to the churn target.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

project_root = Path.cwd().resolve()
if not (project_root / "data").exists():
    project_root = project_root.parent

feature_frame_path = project_root / "data" / "processed" / "feature_frame.parquet"
feature_metadata_path = project_root / "data" / "processed" / "feature_metadata.json"

print("Project root:", project_root)
print("Feature frame exists:", feature_frame_path.exists())
print("Feature metadata exists:", feature_metadata_path.exists())

## 1. Load feature artifacts

Load the engineered feature frame and the saved metadata summary. The metadata gives a fast overview of feature families before we inspect rows directly.

In [ ]:
feature_frame = pd.read_parquet(feature_frame_path)
feature_metadata = json.loads(feature_metadata_path.read_text(encoding="utf-8"))

print("Shape:", feature_frame.shape)
print("")
print("Columns:", feature_frame.shape[1])
print("Rows:", feature_frame.shape[0])
print("")
print("Target distribution:")
print(feature_frame["is_churn"].value_counts(dropna=False).sort_index())
print("")
print("First 25 columns:")
print(feature_frame.columns[:25].tolist())

## 2. Feature groups

Review the engineered features by type. This helps confirm that the output contains the expected mix of numeric, categorical, and datetime features.

In [ ]:
for group_name in ["numeric_features", "categorical_features", "datetime_features", "derived_features"]:
    values = feature_metadata.get(group_name, [])
    print(f"{group_name} ({len(values)}):")
    print(values)
    print("-" * 80)

preview_columns = [
    column
    for column in [
        "is_churn",
        "bd",
        "bd_clean",
        "age_band",
        "gender",
        "gender_clean",
        "profile_completeness",
        "trans_count",
        "total_spend",
        "spend_per_transaction",
        "active_days",
        "total_secs",
        "listen_events_total",
        "usage_to_spend_ratio",
        "recent_transaction_flag",
        "recent_usage_flag",
    ]
    if column in feature_frame.columns
]

feature_frame[preview_columns].head(10)

## 3. Quick quality checks

Check missingness, duplicate rows, and a small set of summary statistics so we can spot issues before modeling.

In [ ]:
missing_rate = feature_frame.isna().mean().sort_values(ascending=False)
missing_summary = missing_rate[missing_rate > 0].to_frame("missing_rate")
missing_summary["missing_count"] = feature_frame.isna().sum()
missing_summary.head(20)

In [ ]:
duplicate_rows = int(feature_frame.duplicated().sum())
duplicate_msno = int(feature_frame["msno"].duplicated().sum()) if "msno" in feature_frame.columns else 0

print("Duplicate rows:", duplicate_rows)
print("Duplicate msno values:", duplicate_msno)

numeric_overview = feature_frame.select_dtypes(include="number").describe().T
numeric_overview.head(20)

## 4. Target view by selected features

Look at a few high-value features against the churn target to see whether the engineered data is already separating churned and non-churned users.

In [ ]:
def target_summary(frame: pd.DataFrame, feature: str) -> pd.DataFrame:
    return (
        frame.groupby(feature, dropna=False)["is_churn"]
        .agg(["count", "mean"])
        .rename(columns={"count": "rows", "mean": "churn_rate"})
        .sort_values(["rows", "churn_rate"], ascending=[False, False])
    )

for feature in ["age_band", "gender_clean", "recent_transaction_flag", "recent_usage_flag"]:
    if feature in feature_frame.columns:
        print(f"### {feature}")
        display(target_summary(feature_frame, feature))
        print()

## 5. Next steps

The feature set is now ready for modeling. The next stage should focus on preprocessing, encoding, train/validation split design, and the baseline logistic regression model.